In [2]:
from pathlib import Path

data_dir = Path(
    "../data/raw/amazon/almrrc2021-data-training/model_build_inputs"
)

for file in sorted(data_dir.iterdir()):
    if file.is_file():
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"{file.name}: {size_mb:.1f} MB")

actual_sequences.json: 9.2 MB
invalid_sequence_scores.json: 0.4 MB
package_data.json: 358.0 MB
readme.md: 0.0 MB
route_data.json: 75.3 MB
travel_times.json: 1733.0 MB


In [3]:
import json
import pandas as pd

route_file = data_dir / "route_data.json"

with route_file.open(encoding="utf-8") as file:
    route_json = json.load(file)

route_rows = []

for route_id, route in route_json.items():
    stops = route["stops"]

    route_rows.append({
        "route_id": route_id,
        "station_code": route["station_code"],
        "date": route["date_YYYY_MM_DD"],
        "departure_time_utc": route["departure_time_utc"],
        "route_score": route["route_score"],
        "vehicle_capacity_cm3": route["executor_capacity_cm3"],
        "stop_count": len(stops),
        "dropoff_count": sum(
            stop["type"] == "Dropoff" for stop in stops.values()
        ),
    })

routes_df = pd.DataFrame(route_rows)

print("Route summary shape:", routes_df.shape)
display(routes_df.head())
display(routes_df["route_score"].value_counts())
display(routes_df[["stop_count", "dropoff_count", "vehicle_capacity_cm3"]].describe().round(1))
display(routes_df.isna().sum())

Route summary shape: (6112, 8)


,route_id,station_code,date,departure_time_utc,route_score,vehicle_capacity_cm3,stop_count,dropoff_count
0,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,DLA3,2018-07-27,16:02:10,High,3313071.0,119,118
1,RouteID_0016bc70-cb8d-48b0-aa55-8ee50bdcdb59,DSE4,2018-07-28,15:44:41,High,4247527.0,106,105
2,RouteID_001948e9-4675-486d-9ec5-912fd8e0770f,DSE5,2018-08-18,15:32:04,High,4247527.0,128,127
3,RouteID_001b4ee3-c4f2-467f-932b-c85524d1021f,DLA9,2018-08-15,15:09:38,High,3313071.0,142,141
4,RouteID_0021a2aa-780f-460d-b09a-f301709e2523,DLA7,2018-08-05,14:23:26,High,3313071.0,155,154


route_score
Medium    3292
High      2718
Low        102
Name: count, dtype: int64

,stop_count,dropoff_count,vehicle_capacity_cm3
count,6112.0,6112.0,6112.0
mean,148.0,147.0,3607178.0
std,31.0,31.0,434472.2
min,33.0,32.0,3114853.2
25%,129.0,128.0,3313071.0
50%,151.0,150.0,3313071.0
75%,170.0,169.0,4247527.0
max,238.0,237.0,4247527.0


route_id                0
station_code            0
date                    0
departure_time_utc      0
route_score             0
vehicle_capacity_cm3    0
stop_count              0
dropoff_count           0
dtype: int64

In [4]:
from collections import Counter

stop_types = Counter()
zones = set()
missing_coordinates = 0
invalid_coordinates = 0
station_counts = []

for route in route_json.values():
    route_station_count = 0

    for stop in route["stops"].values():
        stop_types[stop["type"]] += 1
        zones.add(stop.get("zone_id"))

        lat = stop.get("lat")
        lng = stop.get("lng")

        if lat is None or lng is None:
            missing_coordinates += 1
        elif not (-90 <= lat <= 90 and -180 <= lng <= 180):
            invalid_coordinates += 1

        if stop["type"] == "Station":
            route_station_count += 1

    station_counts.append(route_station_count)

print("Stop types:", stop_types)
print("Unique zones:", len(zones))
print("Routes with station count other than 1:",
      sum(count != 1 for count in station_counts))
print("Missing coordinates:", missing_coordinates)
print("Invalid coordinates:", invalid_coordinates)

Stop types: Counter({'Dropoff': 898415, 'Station': 6112})
Unique zones: 8963
Routes with station count other than 1: 0
Missing coordinates: 0
Invalid coordinates: 0


In [5]:
import json

sequence_file = data_dir / "actual_sequences.json"

with sequence_file.open(encoding="utf-8") as file:
    actual_sequences = json.load(file)

missing_routes = set(route_json) - set(actual_sequences)
extra_routes = set(actual_sequences) - set(route_json)

coverage_issues = []
duplicate_position_routes = []

for route_id, route in route_json.items():
    sequence = actual_sequences.get(route_id, {}).get("actual")

    if sequence is None:
        coverage_issues.append((route_id, "missing sequence"))
        continue

    dropoff_ids = {
        stop_id
        for stop_id, stop in route["stops"].items()
        if stop["type"] == "Dropoff"
    }

    if set(sequence) != dropoff_ids:
        coverage_issues.append((
            route_id,
            f"sequence stops: {len(sequence)}, dropoffs: {len(dropoff_ids)}"
        ))

    positions = list(sequence.values())
    if len(positions) != len(set(positions)):
        duplicate_position_routes.append(route_id)

print("Routes in route data:", len(route_json))
print("Routes in sequence data:", len(actual_sequences))
print("Missing sequence routes:", len(missing_routes))
print("Extra sequence routes:", len(extra_routes))
print("Route/stop coverage issues:", len(coverage_issues))
print("Routes with duplicate sequence positions:", len(duplicate_position_routes))

if coverage_issues:
    print("Examples:", coverage_issues[:5])

Routes in route data: 6112
Routes in sequence data: 6112
Missing sequence routes: 0
Extra sequence routes: 0
Route/stop coverage issues: 6112
Routes with duplicate sequence positions: 0
Examples: [('RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77', 'sequence stops: 119, dropoffs: 118'), ('RouteID_0016bc70-cb8d-48b0-aa55-8ee50bdcdb59', 'sequence stops: 106, dropoffs: 105'), ('RouteID_001948e9-4675-486d-9ec5-912fd8e0770f', 'sequence stops: 128, dropoffs: 127'), ('RouteID_001b4ee3-c4f2-467f-932b-c85524d1021f', 'sequence stops: 142, dropoffs: 141'), ('RouteID_0021a2aa-780f-460d-b09a-f301709e2523', 'sequence stops: 155, dropoffs: 154')]


In [6]:
from collections import Counter

coverage_issues = []
station_positions = Counter()

for route_id, route in route_json.items():
    sequence = actual_sequences[route_id]["actual"]
    all_stop_ids = set(route["stops"])

    if set(sequence) != all_stop_ids:
        coverage_issues.append(route_id)

    station_id = next(
        stop_id
        for stop_id, stop in route["stops"].items()
        if stop["type"] == "Station"
    )
    station_positions[sequence[station_id]] += 1

print("Routes with sequence/stop mismatches:", len(coverage_issues))
print("Station sequence positions:", station_positions)

Routes with sequence/stop mismatches: 0
Station sequence positions: Counter({0: 6112})


In [7]:
import json

package_file = data_dir / "package_data.json"

with package_file.open(encoding="utf-8") as file:
    prefix = file.read(5_000_000)

decoder = json.JSONDecoder()
route_id, position = decoder.raw_decode(prefix, 1)
position = prefix.index(":", position) + 1

while prefix[position].isspace():
    position += 1

packages_by_stop, _ = decoder.raw_decode(prefix, position)

route = route_json[route_id]
dropoff_ids = {
    stop_id
    for stop_id, stop in route["stops"].items()
    if stop["type"] == "Dropoff"
}

first_stop_id, packages = next(iter(packages_by_stop.items()))
first_package_id, package_details = next(iter(packages.items()))

print("Sample route:", route_id)
print("Package-data stops:", len(packages_by_stop))
print("Dropoff stops in route:", len(dropoff_ids))
print("Package stops match route dropoffs:",
      set(packages_by_stop) == dropoff_ids)
print("Example package fields:", package_details)

Sample route: RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77
Package-data stops: 119
Dropoff stops in route: 118
Package stops match route dropoffs: False
Example package fields: {'scan_status': 'DELIVERED', 'time_window': {'start_time_utc': nan, 'end_time_utc': nan}, 'planned_service_time_seconds': 59.3, 'dimensions': {'depth_cm': 25.4, 'height_cm': 7.6, 'width_cm': 17.8}}


In [8]:
package_stop_ids = set(packages_by_stop)
all_route_stop_ids = set(route["stops"])

print("Package stops equal all route stops:",
      package_stop_ids == all_route_stop_ids)
print("In package data only:", package_stop_ids - all_route_stop_ids)
print("In route data only:", all_route_stop_ids - package_stop_ids)

for stop_id in sorted(package_stop_ids | all_route_stop_ids):
    if stop_id not in package_stop_ids or stop_id not in all_route_stop_ids:
        print(
            stop_id,
            "route type:", route["stops"].get(stop_id, {}).get("type"),
            "packages:", len(packages_by_stop.get(stop_id, {}))
        )

Package stops equal all route stops: True
In package data only: set()
In route data only: set()


In [9]:
package_rows = []

for stop_id, packages in packages_by_stop.items():
    for package_id, details in packages.items():
        dimensions = details.get("dimensions", {})
        time_window = details.get("time_window", {})

        has_dimensions = all(
            key in dimensions
            for key in ("depth_cm", "height_cm", "width_cm")
        )

        volume_cm3 = (
            dimensions["depth_cm"]
            * dimensions["height_cm"]
            * dimensions["width_cm"]
            if has_dimensions
            else None
        )

        package_rows.append({
            "stop_id": stop_id,
            "package_id": package_id,
            "scan_status": details.get("scan_status"),
            "has_time_window": (
                isinstance(time_window.get("start_time_utc"), str)
                and isinstance(time_window.get("end_time_utc"), str)
            ),
            "service_seconds": details.get("planned_service_time_seconds"),
            "volume_cm3": volume_cm3,
        })

packages_df = pd.DataFrame(package_rows)

print("Packages on sample route:", len(packages_df))
print("Packages with time windows:",
      packages_df["has_time_window"].sum())
print("Missing service times:",
      packages_df["service_seconds"].isna().sum())
print("Missing package volumes:",
      packages_df["volume_cm3"].isna().sum())
print("Estimated package volume (cm³):",
      round(packages_df["volume_cm3"].sum(), 1))
print("Vehicle capacity (cm³):",
      route["executor_capacity_cm3"])
print("Volume / capacity:",
      round(packages_df["volume_cm3"].sum()
            / route["executor_capacity_cm3"], 3))
display(packages_df["scan_status"].value_counts(dropna=False))

Packages on sample route: 276
Packages with time windows: 55
Missing service times: 0
Missing package volumes: 0
Estimated package volume (cm³): 3032336.7
Vehicle capacity (cm³): 3313071.0
Volume / capacity: 0.915


scan_status
DELIVERED             274
DELIVERY_ATTEMPTED      2
Name: count, dtype: int64

In [10]:
travel_file = data_dir / "travel_times.json"

with travel_file.open(encoding="utf-8") as file:
    prefix = file.read(5_000_000)

decoder = json.JSONDecoder()
travel_route_id, position = decoder.raw_decode(prefix, 1)
position = prefix.index(":", position) + 1

while prefix[position].isspace():
    position += 1

travel_matrix, _ = decoder.raw_decode(prefix, position)

route_stop_ids = set(route_json[travel_route_id]["stops"])
matrix_stop_ids = set(travel_matrix)

row_mismatches = sum(
    set(row) != matrix_stop_ids
    for row in travel_matrix.values()
)

diagonal_is_zero = all(
    row.get(origin) == 0
    for origin, row in travel_matrix.items()
)

travel_values = [
    value
    for row in travel_matrix.values()
    for value in row.values()
]

print("Sample route matches:", travel_route_id == route_id)
print("Matrix origins match route stops:", matrix_stop_ids == route_stop_ids)
print("Rows with destination mismatches:", row_mismatches)
print("Diagonal travel times are zero:", diagonal_is_zero)
print("Travel-time range (seconds):",
      round(min(travel_values), 1), "to", round(max(travel_values), 1))

Sample route matches: True
Matrix origins match route stops: True
Rows with destination mismatches: 0
Diagonal travel times are zero: True
Travel-time range (seconds): 0.0 to 2011.9


In [11]:
sequence = actual_sequences[route_id]["actual"]

ordered_stop_ids = [
    stop_id
    for stop_id, position in sorted(
        sequence.items(),
        key=lambda item: item[1]
    )
]

baseline_travel_seconds = sum(
    travel_matrix[origin][destination]
    for origin, destination in zip(
        ordered_stop_ids,
        ordered_stop_ids[1:]
    )
)

baseline_service_seconds = sum(
    details["planned_service_time_seconds"]
    for packages in packages_by_stop.values()
    for details in packages.values()
)

print("Route score:", route["route_score"])
print("Stops in sequence:", len(ordered_stop_ids))
print("Sequence starts at station:",
      route["stops"][ordered_stop_ids[0]]["type"] == "Station")
print("Estimated travel time (minutes):",
      round(baseline_travel_seconds / 60, 1))
print("Planned service time (minutes):",
      round(baseline_service_seconds / 60, 1))
print("Estimated total route time (minutes):",
      round((baseline_travel_seconds + baseline_service_seconds) / 60, 1))

Route score: High
Stops in sequence: 119
Sequence starts at station: True
Estimated travel time (minutes): 129.2
Planned service time (minutes): 215.7
Estimated total route time (minutes): 344.9


In [12]:
from datetime import datetime
from statistics import median

window_durations = []
partial_windows = 0
invalid_windows = 0

for packages in packages_by_stop.values():
    for details in packages.values():
        window = details.get("time_window", {})
        start = window.get("start_time_utc")
        end = window.get("end_time_utc")

        has_start = isinstance(start, str)
        has_end = isinstance(end, str)

        if has_start and has_end:
            start_dt = datetime.fromisoformat(start)
            end_dt = datetime.fromisoformat(end)

            if end_dt < start_dt:
                invalid_windows += 1
            else:
                window_durations.append(
                    (end_dt - start_dt).total_seconds() / 60
                )
        elif has_start or has_end:
            partial_windows += 1

print("Complete time windows:", len(window_durations))
print("Partially specified windows:", partial_windows)
print("Invalid windows:", invalid_windows)

if window_durations:
    print("Window duration in minutes — min / median / max:",
          round(min(window_durations), 1),
          round(median(window_durations), 1),
          round(max(window_durations), 1))

Complete time windows: 55
Partially specified windows: 0
Invalid windows: 0
Window duration in minutes — min / median / max: 240.0 420.0 660.0


In [13]:
from collections import defaultdict
from datetime import datetime

windows_by_stop = defaultdict(list)

for stop_id, packages in packages_by_stop.items():
    for details in packages.values():
        window = details.get("time_window", {})
        start = window.get("start_time_utc")
        end = window.get("end_time_utc")

        if isinstance(start, str) and isinstance(end, str):
            windows_by_stop[stop_id].append((
                datetime.fromisoformat(start),
                datetime.fromisoformat(end)
            ))

shared_window_stops = 0
nonoverlapping_stops = []

for stop_id, windows in windows_by_stop.items():
    latest_start = max(start for start, end in windows)
    earliest_end = min(end for start, end in windows)

    if latest_start <= earliest_end:
        shared_window_stops += 1
    else:
        nonoverlapping_stops.append(stop_id)

print("Stops with at least one windowed package:", len(windows_by_stop))
print("Stops with a common shared window:", shared_window_stops)
print("Stops without a common shared window:", len(nonoverlapping_stops))
print("Examples without a common window:", nonoverlapping_stops[:5])

Stops with at least one windowed package: 30
Stops with a common shared window: 30
Stops without a common shared window: 0
Examples without a common window: []


In [14]:
import numpy as np

stop_ids = list(sample_matrix)
asymmetries = []

for origin in stop_ids:
    for destination in stop_ids:
        if origin >= destination:
            continue

        forward = sample_matrix[origin][destination]
        reverse = sample_matrix[destination][origin]

        if not np.isclose(forward, reverse, atol=0.1):
            asymmetries.append((origin, destination, forward, reverse))

print("Compared stop pairs:", len(stop_ids) * (len(stop_ids) - 1) // 2)
print("Pairs with different travel times:", len(asymmetries))
print("Example differences:", asymmetries[:5])

NameError: name 'sample_matrix' is not defined

In [15]:
[
    (name, type(value).__name__)
    for name, value in globals().items()
    if "matrix" in name.lower() or "travel" in name.lower()
]

[('travel_file', 'PosixPath'),
 ('travel_route_id', 'str'),
 ('travel_matrix', 'dict'),
 ('matrix_stop_ids', 'set'),
 ('travel_values', 'list'),
 ('baseline_travel_seconds', 'float')]

In [16]:
import numpy as np

stop_ids = list(travel_matrix)
asymmetries = []

for i, origin in enumerate(stop_ids):
    for destination in stop_ids[i + 1:]:
        forward = travel_matrix[origin][destination]
        reverse = travel_matrix[destination][origin]

        if not np.isclose(forward, reverse, atol=0.1):
            asymmetries.append((origin, destination, forward, reverse))

print("Compared stop pairs:", len(stop_ids) * (len(stop_ids) - 1) // 2)
print("Pairs with different travel times:", len(asymmetries))
print("Example differences:", asymmetries[:5])

Compared stop pairs: 7021
Pairs with different travel times: 6964
Example differences: [('AD', 'AF', 198.3, 209.8), ('AD', 'AG', 264.9, 295.7), ('AD', 'BA', 268.9, 284.0), ('AD', 'BE', 89.4, 107.9), ('AD', 'BG', 140.0, 136.3)]


In [19]:
sample_route_ids = routes_df["route_id"].iloc[:20].tolist()
sample_route_ids

['RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77',
 'RouteID_0016bc70-cb8d-48b0-aa55-8ee50bdcdb59',
 'RouteID_001948e9-4675-486d-9ec5-912fd8e0770f',
 'RouteID_001b4ee3-c4f2-467f-932b-c85524d1021f',
 'RouteID_0021a2aa-780f-460d-b09a-f301709e2523',
 'RouteID_0036cf37-1c93-48f8-b337-3dd6d1a195d8',
 'RouteID_00437423-5a8f-4fac-8efa-85264f0a3944',
 'RouteID_00573a3d-b909-42ee-8a99-c6474309d96b',
 'RouteID_00575a14-7d19-4291-8853-8f7095ed1bc6',
 'RouteID_00575ca4-8a63-49d2-96c8-9b347be5ba6c',
 'RouteID_00595235-52e7-4b68-883c-70482701dea0',
 'RouteID_006f581c-3087-4edf-8802-278b078157b9',
 'RouteID_00747543-3f2f-47ae-a2f2-91b30ce207ab',
 'RouteID_0074e307-23c1-4b15-afa2-e58b45e4403c',
 'RouteID_0076d53c-6bfa-4e22-b39e-08769fb8e4d3',
 'RouteID_007a8397-37f4-4775-a452-c6934167526b',
 'RouteID_007b8544-11dc-47b2-be61-0095f8f2af71',
 'RouteID_0091a35c-85fa-4b5e-90cf-4778f441753e',
 'RouteID_00930ddc-25f9-4371-a437-1e37c9d0a80d',
 'RouteID_00a25e47-65c9-4efc-bfe9-8d9648a0df4e']

In [20]:
import importlib.util

print(importlib.util.find_spec("ijson"))

None


In [22]:
import importlib.util

print(importlib.util.find_spec("ijson"))

ModuleSpec(name='ijson', loader=<_frozen_importlib_external.SourceFileLoader object at 0x103274170>, origin='/Users/harshitbhargava/Desktop/LMD Project/.venv/lib/python3.13/site-packages/ijson/__init__.py', submodule_search_locations=['/Users/harshitbhargava/Desktop/LMD Project/.venv/lib/python3.13/site-packages/ijson'])


In [23]:
import ijson

sample_route_id_set = set(sample_route_ids)
package_audit = {}

with package_file.open("rb") as file:
    for route_id, packages_by_stop in ijson.kvitems(file, ""):
        if route_id not in sample_route_id_set:
            continue

        route_stop_ids = set(route_json[route_id]["stops"])
        package_stop_ids = set(packages_by_stop)

        package_audit[route_id] = {
            "route_stop_count": len(route_stop_ids),
            "package_stop_count": len(package_stop_ids),
            "missing_stop_ids": route_stop_ids - package_stop_ids,
            "extra_stop_ids": package_stop_ids - route_stop_ids,
            "package_count": sum(
                len(packages) for packages in packages_by_stop.values()
            ),
        }

        if len(package_audit) == len(sample_route_id_set):
            break

print("Sample routes requested:", len(sample_route_id_set))
print("Routes found in package data:", len(package_audit))

for route_id, result in package_audit.items():
    print(
        route_id,
        "| route stops:", result["route_stop_count"],
        "| package stops:", result["package_stop_count"],
        "| packages:", result["package_count"],
        "| missing stops:", len(result["missing_stop_ids"]),
        "| extra stops:", len(result["extra_stop_ids"]),
    )

IncompleteJSONError: lexical error: invalid char in json text.
          _window": {"start_time_utc": NaN, "end_time_utc": NaN}, "pla
                     (right here) ------^


In [24]:
import json

def iter_json_object_items(file, chunk_size=1_000_000):
    decoder = json.JSONDecoder(parse_constant=lambda value: None)
    buffer = ""
    eof = False

    def read_more():
        nonlocal buffer, eof
        chunk = file.read(chunk_size)
        if chunk:
            buffer += chunk
        else:
            eof = True

    def skip_whitespace(position):
        while position < len(buffer) and buffer[position].isspace():
            position += 1
        return position

    def decode_at(position):
        while True:
            try:
                return decoder.raw_decode(buffer, position)
            except json.JSONDecodeError:
                if eof:
                    raise
                read_more()

    read_more()
    position = skip_whitespace(0)

    if position >= len(buffer) or buffer[position] != "{":
        raise ValueError("Expected a JSON object at the start of the file.")

    buffer = buffer[position + 1:]

    while True:
        position = skip_whitespace(0)

        if position >= len(buffer):
            if eof:
                raise ValueError("Unexpected end of the JSON file.")
            read_more()
            continue

        if buffer[position] == "}":
            break

        if buffer[position] == ",":
            position = skip_whitespace(position + 1)

        route_id, position = decode_at(position)
        position = skip_whitespace(position)

        if position >= len(buffer) or buffer[position] != ":":
            if eof:
                raise ValueError("Expected ':' after a route ID.")
            read_more()
            continue

        packages_by_stop, position = decode_at(position + 1)

        buffer = buffer[position:]
        yield route_id, packages_by_stop


sample_route_id_set = set(sample_route_ids)
package_audit = {}

with package_file.open("r", encoding="utf-8") as file:
    for route_id, packages_by_stop in iter_json_object_items(file):
        if route_id not in sample_route_id_set:
            continue

        route_stop_ids = set(route_json[route_id]["stops"])
        package_stop_ids = set(packages_by_stop)

        package_audit[route_id] = {
            "route_stop_count": len(route_stop_ids),
            "package_stop_count": len(package_stop_ids),
            "missing_stop_ids": route_stop_ids - package_stop_ids,
            "extra_stop_ids": package_stop_ids - route_stop_ids,
            "package_count": sum(
                len(packages) for packages in packages_by_stop.values()
            ),
        }

        if len(package_audit) == len(sample_route_id_set):
            break

print("Sample routes requested:", len(sample_route_id_set))
print("Routes found in package data:", len(package_audit))

for route_id, result in package_audit.items():
    print(
        route_id,
        "| route stops:", result["route_stop_count"],
        "| package stops:", result["package_stop_count"],
        "| packages:", result["package_count"],
        "| missing stops:", len(result["missing_stop_ids"]),
        "| extra stops:", len(result["extra_stop_ids"]),
    )

JSONDecodeError: Expecting value: line 1 column 48 (char 47)

In [25]:
import json

def iter_json_object_items(file, chunk_size=1_000_000):
    decoder = json.JSONDecoder(parse_constant=lambda value: None)
    buffer = ""
    eof = False

    def read_more():
        nonlocal buffer, eof
        chunk = file.read(chunk_size)
        if chunk:
            buffer += chunk
        else:
            eof = True

    def skip_whitespace(position):
        while position < len(buffer) and buffer[position].isspace():
            position += 1
        return position

    def decode_at(position):
        while True:
            try:
                return decoder.raw_decode(buffer, position)
            except json.JSONDecodeError:
                if eof:
                    raise
                read_more()

    read_more()
    position = skip_whitespace(0)

    if position >= len(buffer) or buffer[position] != "{":
        raise ValueError("Expected a JSON object at the start of the file.")

    buffer = buffer[position + 1:]

    while True:
        position = skip_whitespace(0)

        if position >= len(buffer):
            if eof:
                raise ValueError("Unexpected end of the JSON file.")
            read_more()
            continue

        if buffer[position] == "}":
            break

        if buffer[position] == ",":
            position = skip_whitespace(position + 1)

        route_id, position = decode_at(position)
        position = skip_whitespace(position)

        if position >= len(buffer) or buffer[position] != ":":
            if eof:
                raise ValueError("Expected ':' after a route ID.")
            read_more()
            continue

        position = skip_whitespace(position + 1)
        packages_by_stop, position = decode_at(position)

        buffer = buffer[position:]
        yield route_id, packages_by_stop


sample_route_id_set = set(sample_route_ids)
package_audit = {}

with package_file.open("r", encoding="utf-8") as file:
    for route_id, packages_by_stop in iter_json_object_items(file):
        if route_id not in sample_route_id_set:
            continue

        route_stop_ids = set(route_json[route_id]["stops"])
        package_stop_ids = set(packages_by_stop)

        package_audit[route_id] = {
            "route_stop_count": len(route_stop_ids),
            "package_stop_count": len(package_stop_ids),
            "missing_stop_ids": route_stop_ids - package_stop_ids,
            "extra_stop_ids": package_stop_ids - route_stop_ids,
            "package_count": sum(
                len(packages) for packages in packages_by_stop.values()
            ),
        }

        if len(package_audit) == len(sample_route_id_set):
            break

print("Sample routes requested:", len(sample_route_id_set))
print("Routes found in package data:", len(package_audit))

for route_id, result in package_audit.items():
    print(
        route_id,
        "| route stops:", result["route_stop_count"],
        "| package stops:", result["package_stop_count"],
        "| packages:", result["package_count"],
        "| missing stops:", len(result["missing_stop_ids"]),
        "| extra stops:", len(result["extra_stop_ids"]),
    )

Sample routes requested: 20
Routes found in package data: 20
RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77 | route stops: 119 | package stops: 119 | packages: 276 | missing stops: 0 | extra stops: 0
RouteID_0016bc70-cb8d-48b0-aa55-8ee50bdcdb59 | route stops: 106 | package stops: 106 | packages: 199 | missing stops: 0 | extra stops: 0
RouteID_001948e9-4675-486d-9ec5-912fd8e0770f | route stops: 128 | package stops: 128 | packages: 212 | missing stops: 0 | extra stops: 0
RouteID_001b4ee3-c4f2-467f-932b-c85524d1021f | route stops: 142 | package stops: 142 | packages: 241 | missing stops: 0 | extra stops: 0
RouteID_0021a2aa-780f-460d-b09a-f301709e2523 | route stops: 155 | package stops: 155 | packages: 219 | missing stops: 0 | extra stops: 0
RouteID_0036cf37-1c93-48f8-b337-3dd6d1a195d8 | route stops: 190 | package stops: 190 | packages: 276 | missing stops: 0 | extra stops: 0
RouteID_00437423-5a8f-4fac-8efa-85264f0a3944 | route stops: 161 | package stops: 161 | packages: 216 | missing stops:

In [26]:
from collections import defaultdict
from datetime import datetime
from statistics import median

package_field_results = []
window_durations = []

with package_file.open("r", encoding="utf-8") as file:
    for route_id, packages_by_stop in iter_json_object_items(file):
        if route_id not in sample_route_id_set:
            continue

        result = {
            "route_id": route_id,
            "package_count": 0,
            "missing_service_times": 0,
            "packages_missing_dimensions": 0,
            "complete_valid_windows": 0,
            "partial_windows": 0,
            "absent_windows": 0,
            "invalid_windows": 0,
        }

        windows_by_stop = defaultdict(list)

        for stop_id, packages in packages_by_stop.items():
            for package_id, details in packages.items():
                result["package_count"] += 1

                # Check: package ka planned service time diya gaya hai?
                service_time = details.get("planned_service_time_seconds")
                if service_time is None:
                    result["missing_service_times"] += 1

                # Check: teeno dimensions maujood aur non-empty hain?
                dimensions = details.get("dimensions") or {}
                dimension_names = ("depth_cm", "height_cm", "width_cm")

                has_all_dimensions = all(
                    dimensions.get(name) is not None
                    for name in dimension_names
                )

                if not has_all_dimensions:
                    result["packages_missing_dimensions"] += 1

                # Check: time window absent, partial, complete, ya invalid hai?
                time_window = details.get("time_window") or {}
                start = time_window.get("start_time_utc")
                end = time_window.get("end_time_utc")

                has_start = isinstance(start, str)
                has_end = isinstance(end, str)

                if not has_start and not has_end:
                    result["absent_windows"] += 1

                elif has_start != has_end:
                    result["partial_windows"] += 1

                else:
                    try:
                        start_dt = datetime.fromisoformat(start)
                        end_dt = datetime.fromisoformat(end)

                        if end_dt < start_dt:
                            result["invalid_windows"] += 1
                        else:
                            result["complete_valid_windows"] += 1
                            window_durations.append(
                                (end_dt - start_dt).total_seconds() / 60
                            )
                            windows_by_stop[stop_id].append(
                                (start_dt, end_dt)
                            )

                    except ValueError:
                        result["invalid_windows"] += 1

        # Har stop ke packages ki valid windows mein common overlap check karo.
        stops_with_windows = 0
        stops_with_common_window = 0

        for windows in windows_by_stop.values():
            stops_with_windows += 1

            latest_start = max(start for start, end in windows)
            earliest_end = min(end for start, end in windows)

            if latest_start <= earliest_end:
                stops_with_common_window += 1

        result["stops_with_windows"] = stops_with_windows
        result["stops_with_common_window"] = stops_with_common_window
        result["stops_without_common_window"] = (
            stops_with_windows - stops_with_common_window
        )

        package_field_results.append(result)

        if len(package_field_results) == len(sample_route_id_set):
            break

package_fields_df = pd.DataFrame(package_field_results)

print("Routes requested:", len(sample_route_id_set))
print("Routes audited:", len(package_fields_df))
print("Packages audited:", package_fields_df["package_count"].sum())
print()

display(package_fields_df)

print("Overall field and window totals:")
display(
    package_fields_df[
        [
            "package_count",
            "missing_service_times",
            "packages_missing_dimensions",
            "complete_valid_windows",
            "partial_windows",
            "absent_windows",
            "invalid_windows",
            "stops_with_windows",
            "stops_with_common_window",
            "stops_without_common_window",
        ]
    ].sum().to_frame("total")
)

if window_durations:
    print(
        "Valid window duration in minutes — min / median / max:",
        round(min(window_durations), 1),
        round(median(window_durations), 1),
        round(max(window_durations), 1),
    )

Routes requested: 20
Routes audited: 20
Packages audited: 4734



,route_id,package_count,missing_service_times,packages_missing_dimensions,complete_valid_windows,partial_windows,absent_windows,invalid_windows,stops_with_windows,stops_with_common_window,stops_without_common_window
0,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,276,0,0,55,0,221,0,30,30,0
1,RouteID_0016bc70-cb8d-48b0-aa55-8ee50bdcdb59,199,0,0,17,0,182,0,1,1,0
2,RouteID_001948e9-4675-486d-9ec5-912fd8e0770f,212,0,0,25,0,187,0,3,3,0
3,RouteID_001b4ee3-c4f2-467f-932b-c85524d1021f,241,0,0,21,0,220,0,7,7,0
4,RouteID_0021a2aa-780f-460d-b09a-f301709e2523,219,0,0,1,0,218,0,1,1,0
5,RouteID_0036cf37-1c93-48f8-b337-3dd6d1a195d8,276,0,0,9,0,267,0,6,6,0
6,RouteID_00437423-5a8f-4fac-8efa-85264f0a3944,216,0,0,6,0,210,0,4,4,0
7,RouteID_00573a3d-b909-42ee-8a99-c6474309d96b,196,0,0,19,0,177,0,12,12,0
8,RouteID_00575a14-7d19-4291-8853-8f7095ed1bc6,268,0,0,1,0,267,0,1,1,0
9,RouteID_00575ca4-8a63-49d2-96c8-9b347be5ba6c,211,0,0,77,0,134,0,16,16,0


Overall field and window totals:


,total
package_count,4734
missing_service_times,0
packages_missing_dimensions,0
complete_valid_windows,366
partial_windows,0
absent_windows,4368
invalid_windows,0
stops_with_windows,170
stops_with_common_window,170
stops_without_common_window,0


Valid window duration in minutes — min / median / max: 240.0 480.0 1439.0


In [27]:
import math

travel_audit_results = []
sample_route_id_set = set(sample_route_ids)

with travel_file.open("r", encoding="utf-8") as file:
    for route_id, travel_matrix in iter_json_object_items(file):
        if route_id not in sample_route_id_set:
            continue

        expected_stop_ids = set(route_json[route_id]["stops"])
        matrix_origin_ids = set(travel_matrix)

        missing_origins = expected_stop_ids - matrix_origin_ids
        extra_origins = matrix_origin_ids - expected_stop_ids

        missing_destinations = 0
        extra_destinations = 0
        row_mismatches = 0
        diagonal_zero = True
        valid_times = []
        invalid_time_count = 0
        asymmetric_pairs = 0
        comparable_pairs = 0

        for origin, row in travel_matrix.items():
            row_destinations = set(row)

            missing_destinations += len(expected_stop_ids - row_destinations)
            extra_destinations += len(row_destinations - expected_stop_ids)

            if row_destinations != expected_stop_ids:
                row_mismatches += 1

            if row.get(origin) != 0:
                diagonal_zero = False

            for value in row.values():
                if (
                    isinstance(value, (int, float))
                    and math.isfinite(value)
                    and value >= 0
                ):
                    valid_times.append(value)
                else:
                    invalid_time_count += 1

        matrix_stop_list = list(matrix_origin_ids)

        for i, origin in enumerate(matrix_stop_list):
            for destination in matrix_stop_list[i + 1:]:
                forward = travel_matrix[origin].get(destination)
                reverse = travel_matrix[destination].get(origin)

                both_valid = all(
                    isinstance(value, (int, float))
                    and math.isfinite(value)
                    and value >= 0
                    for value in (forward, reverse)
                )

                if both_valid:
                    comparable_pairs += 1

                    if abs(forward - reverse) > 0.1:
                        asymmetric_pairs += 1

        travel_audit_results.append({
            "route_id": route_id,
            "route_stop_count": len(expected_stop_ids),
            "matrix_origin_count": len(matrix_origin_ids),
            "missing_origins": len(missing_origins),
            "extra_origins": len(extra_origins),
            "rows_with_destination_mismatches": row_mismatches,
            "missing_destinations_total": missing_destinations,
            "extra_destinations_total": extra_destinations,
            "diagonal_zero": diagonal_zero,
            "invalid_travel_times": invalid_time_count,
            "min_travel_seconds": min(valid_times) if valid_times else None,
            "max_travel_seconds": max(valid_times) if valid_times else None,
            "compared_directional_pairs": comparable_pairs,
            "asymmetric_pairs": asymmetric_pairs,
        })

        if len(travel_audit_results) == len(sample_route_id_set):
            break

travel_audit_df = pd.DataFrame(travel_audit_results)

print("Routes requested:", len(sample_route_id_set))
print("Routes audited:", len(travel_audit_df))
display(travel_audit_df)

Routes requested: 20
Routes audited: 20


,route_id,route_stop_count,matrix_origin_count,missing_origins,extra_origins,rows_with_destination_mismatches,missing_destinations_total,extra_destinations_total,diagonal_zero,invalid_travel_times,min_travel_seconds,max_travel_seconds,compared_directional_pairs,asymmetric_pairs
0,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,119,119,0,0,0,0,0,True,0,0.0,2011.9,7021,6974
1,RouteID_0016bc70-cb8d-48b0-aa55-8ee50bdcdb59,106,106,0,0,0,0,0,True,0,0.0,2520.7,5565,5448
2,RouteID_001948e9-4675-486d-9ec5-912fd8e0770f,128,128,0,0,0,0,0,True,0,0.0,2882.6,8128,8060
3,RouteID_001b4ee3-c4f2-467f-932b-c85524d1021f,142,142,0,0,0,0,0,True,0,0.0,2382.3,10011,9863
4,RouteID_0021a2aa-780f-460d-b09a-f301709e2523,155,155,0,0,0,0,0,True,0,0.0,2095.7,11935,11855
5,RouteID_0036cf37-1c93-48f8-b337-3dd6d1a195d8,190,190,0,0,0,0,0,True,0,0.0,2776.4,17955,17741
6,RouteID_00437423-5a8f-4fac-8efa-85264f0a3944,161,161,0,0,0,0,0,True,0,0.0,2268.4,12880,12781
7,RouteID_00573a3d-b909-42ee-8a99-c6474309d96b,155,155,0,0,0,0,0,True,0,0.0,2511.0,11935,11813
8,RouteID_00575a14-7d19-4291-8853-8f7095ed1bc6,177,177,0,0,0,0,0,True,0,0.0,2190.8,15576,15466
9,RouteID_00575ca4-8a63-49d2-96c8-9b347be5ba6c,59,59,0,0,0,0,0,True,0,0.0,2144.6,1711,1680
